<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/03_classifier_training_source_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Classifier training

Trains ResNet18 and EfficientNet-B0 (ImageNet-pretrained, fully fine-tuned) as binary tampered/authentic classifiers. All 1,828 spliced images are held out entirely from training and validation — the classifiers learn tampered-vs-authentic from copy-move tampered images plus authentic images only, so the Grad-CAM evaluation in notebook 04 runs on images the classifiers never saw during training.

## Setup — mount Drive, rebuild validated file lists

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import random
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report
from PIL import Image

random.seed(42)
torch.manual_seed(42)

base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"  # adjust to your actual path
au_dir = os.path.join(base, "Au")
tp_dir = os.path.join(base, "Tp")

with open(os.path.join(base, "au_list.txt")) as f:
    au_list_content = [line.strip() for line in f if line.strip()]
with open(os.path.join(base, "tp_list.txt")) as f:
    tp_list_content = [line.strip() for line in f if line.strip()]

au_files_final = sorted(set(au_list_content) & set(os.listdir(au_dir)))
tp_files_final = sorted(set(tp_list_content) & set(os.listdir(tp_dir)))

print(f"Authentic: {len(au_files_final)} | Tampered: {len(tp_files_final)}")  # expect 7491, 5123

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Mounted at /content/drive
Authentic: 7491 | Tampered: 5123
Device: cuda


## Train/val split — spliced images fully held out
All 1,828 spliced images are excluded from classifier training and validation entirely. The classifier learns tampered-vs-authentic from copy-move + authentic images only; the full spliced set is reserved for notebook 04's Grad-CAM evaluation. Saved to Drive so it can be reproduced exactly across sessions.

In [2]:
spliced_files_all = [f for f in tp_files_final if f.split("_")[1] == "D"]
non_spliced_tampered = [f for f in tp_files_final if f not in set(spliced_files_all)]
print(f"Spliced (held out): {len(spliced_files_all)} | Non-spliced tampered (train/val pool): {len(non_spliced_tampered)}")

# --- Source-image exclusion: remove any authentic image that is a source
# --- of a held-out splice, BEFORE the authentic train/val split happens.
def extract_source_tokens(spliced_filename):
    parts = spliced_filename.split("_")
    return [parts[5], parts[6]]

def extract_au_token(au_filename):
    stem = os.path.splitext(au_filename)[0]
    parts = stem.split("_")
    return "".join(parts[1:])

held_out_source_tokens = set()
for fname in spliced_files_all:
    held_out_source_tokens.update(extract_source_tokens(fname))

au_files_eligible = [f for f in au_files_final if extract_au_token(f) not in held_out_source_tokens]
au_files_excluded = [f for f in au_files_final if extract_au_token(f) in held_out_source_tokens]
print(f"Authentic images excluded (source of a held-out splice): {len(au_files_excluded)}")
print(f"Authentic images remaining, eligible for train/val: {len(au_files_eligible)}")

random.shuffle(non_spliced_tampered)
random.shuffle(au_files_eligible)

def split(files, val_frac=0.2):
    cut = int(len(files) * (1 - val_frac))
    return files[:cut], files[cut:]

train_tp, val_tp = split(non_spliced_tampered)
train_au, val_au = split(au_files_eligible)

split_dict = {"train_tampered": train_tp, "val_tampered": val_tp,
              "train_authentic": train_au, "val_authentic": val_au,
              "held_out_spliced": spliced_files_all,
              "authentic_excluded_source": au_files_excluded, "seed": 42}
with open("/content/drive/MyDrive/CASIA2.0/casia_split.json", "w") as f:
    json.dump(split_dict, f)

print(f"Train: {len(train_tp)+len(train_au)} | Val: {len(val_tp)+len(val_au)}")

Spliced (held out): 1828 | Non-spliced tampered (train/val pool): 3295
Authentic images excluded (source of a held-out splice): 1077
Authentic images remaining, eligible for train/val: 6414
Train: 7767 | Val: 1942


## Leak check — confirm spliced images are disjoint from train/val

In [3]:
assert set(spliced_files_all).isdisjoint(train_tp), "Leak: spliced image in training set"
assert set(spliced_files_all).isdisjoint(val_tp), "Leak: spliced image in validation set"
print("Confirmed: all spliced images are disjoint from classifier train/val.")

Confirmed: all spliced images are disjoint from classifier train/val.


## Source-image leak check — do held-out splices share an authentic source with train/val?

CASIA v2 spliced filenames embed the identities of the two source images used
to build the composite (e.g. `Tp_D_CND_M_N_ani00018_sec00096_00138.tif` was
built from authentic images `ani00018` and `sec00096`). The earlier leak
check above confirms no spliced *file* itself leaked into train/val, but does
not check whether the *authentic source images* underlying a held-out splice
were themselves used as authentic training/validation examples. This checks
that second, distinct possibility.

Authentic filenames use the format `Au_<category>_<number>.<ext>` (e.g.
`Au_ani_00018.jpg`) — note the underscore between category and number, which
the splice-embedded token (`ani00018`) does not have. The token reconstruction
below accounts for this so the two naming conventions can be compared
correctly; a naive string match without this step would silently report zero
overlap even if real overlap exists.

In [4]:
# Verification: after the fix above, this should now show ZERO overlap.
# (held_out_source_tokens, train_au, val_au already reflect the exclusion.)
train_au_tokens = {extract_au_token(f) for f in train_au}
val_au_tokens = {extract_au_token(f) for f in val_au}

overlap_train = held_out_source_tokens & train_au_tokens
overlap_val = held_out_source_tokens & val_au_tokens

print(f"Held-out splice source images still in authentic TRAIN: {len(overlap_train)}")
print(f"Held-out splice source images still in authentic VAL:   {len(overlap_val)}")
assert len(overlap_train) == 0, "Source-image leak into TRAIN still present"
assert len(overlap_val) == 0, "Source-image leak into VAL still present"
print("Confirmed: no source-image overlap between held-out splices and authentic train/val.")

Held-out splice source images still in authentic TRAIN: 0
Held-out splice source images still in authentic VAL:   0
Confirmed: no source-image overlap between held-out splices and authentic train/val.


## Copy images to local disk (speeds up training)
Reading thousands of individual files from Drive per epoch is slow. Copying once to Colab's local disk (`/content/`) speeds up every subsequent epoch. Local disk is wiped if the runtime disconnects — re-run this cell if that happens.

In [5]:
import shutil, time

local_base = "/content/CASIA2.0_local"
os.makedirs(local_base, exist_ok=True)

t0 = time.time()
if not os.path.exists(os.path.join(local_base, "Au")):
    shutil.copytree(au_dir, os.path.join(local_base, "Au"))
if not os.path.exists(os.path.join(local_base, "Tp")):
    shutil.copytree(tp_dir, os.path.join(local_base, "Tp"))
print(f"Copied in {time.time()-t0:.1f}s")

au_dir = os.path.join(local_base, "Au")
tp_dir = os.path.join(local_base, "Tp")

Copied in 528.8s


In [6]:
n_local_au = len(os.listdir("/content/CASIA2.0_local/Au"))
n_local_tp = len(os.listdir("/content/CASIA2.0_local/Tp"))

n_drive_au = len(os.listdir("/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised/Au"))
n_drive_tp = len(os.listdir("/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised/Tp"))

print(f"Au: local={n_local_au}, drive={n_drive_au}, match={n_local_au == n_drive_au}")
print(f"Tp: local={n_local_tp}, drive={n_drive_tp}, match={n_local_tp == n_drive_tp}")

Au: local=7492, drive=7492, match=True
Tp: local=5123, drive=5123, match=True


## Dataset, transforms, and DataLoaders
Augmentation (flip, rotation, color jitter) applied only to training data — validation data is left unaugmented so it reflects real performance.

In [9]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CASIADataset(Dataset):
    def __init__(self, tampered_files, authentic_files, tampered_dir, authentic_dir, transform):
        self.samples = [(os.path.join(tampered_dir, f), 1) for f in tampered_files] + \
                        [(os.path.join(authentic_dir, f), 0) for f in authentic_files]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label

train_dataset = CASIADataset(train_tp, train_au, tp_dir, au_dir, train_transform)
val_dataset = CASIADataset(val_tp, val_au, tp_dir, au_dir, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 243 | Val batches: 61


## Verify batch-load speed before training

In [10]:
t0 = time.time()
images, labels = next(iter(train_loader))
print(f"First batch loaded in {time.time()-t0:.1f}s")

First batch loaded in 0.5s


## Generic training loop
Shared by both architectures below. Early stopping on validation loss (patience=4) prevents chasing epochs past the point of diminishing returns.

In [11]:
def train_model(model, model_name, max_epochs=30, patience=4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
               "val_f1": [], "val_balanced_acc": []}
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_labels, all_preds)
        val_bal_acc = balanced_accuracy_score(all_labels, all_preds)

        scheduler.step(val_loss)
        for k, v in zip(history.keys(), [train_loss, val_loss, train_acc, val_acc, val_f1, val_bal_acc]):
            history[k].append(v)

        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"val_acc={val_acc:.4f} val_f1={val_f1:.4f} val_bal_acc={val_bal_acc:.4f}")

        torch.save(model.state_dict(), f"/content/drive/MyDrive/CASIA2.0/{model_name}_last.pt")
        with open(f"/content/drive/MyDrive/CASIA2.0/{model_name}_history.json", "w") as f:
            json.dump(history, f)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f"/content/drive/MyDrive/CASIA2.0/{model_name}_best.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    return model, history

## Train ResNet18

In [12]:
resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet, resnet_history = train_model(resnet, "casia_resnet")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 238MB/s]


Epoch 1: train_loss=0.5130 val_loss=0.5060 val_acc=0.7951 val_f1=0.6801 val_bal_acc=0.7578
Epoch 2: train_loss=0.3970 val_loss=0.4373 val_acc=0.7878 val_f1=0.7235 val_bal_acc=0.7952
Epoch 3: train_loss=0.3421 val_loss=0.4411 val_acc=0.8157 val_f1=0.7410 val_bal_acc=0.8062
Epoch 4: train_loss=0.3058 val_loss=0.4797 val_acc=0.8043 val_f1=0.6839 val_bal_acc=0.7604
Epoch 5: train_loss=0.2933 val_loss=0.4548 val_acc=0.7915 val_f1=0.7385 val_bal_acc=0.8101
Epoch 6: train_loss=0.2381 val_loss=0.4365 val_acc=0.8265 val_f1=0.7479 val_bal_acc=0.8100
Epoch 7: train_loss=0.2184 val_loss=0.4270 val_acc=0.8244 val_f1=0.7476 val_bal_acc=0.8103
Epoch 8: train_loss=0.2141 val_loss=0.4744 val_acc=0.8239 val_f1=0.7482 val_bal_acc=0.8110
Epoch 9: train_loss=0.2121 val_loss=0.4286 val_acc=0.8363 val_f1=0.7770 val_bal_acc=0.8373
Epoch 10: train_loss=0.2029 val_loss=0.4696 val_acc=0.8249 val_f1=0.7478 val_bal_acc=0.8103
Epoch 11: train_loss=0.1858 val_loss=0.5069 val_acc=0.8187 val_f1=0.7321 val_bal_acc=0.79

## Train EfficientNet-B0
Target layer differs (`features[-1]`, not `layer4`) — handled in notebook 04, not here.

In [14]:
effnet = models.efficientnet_b0(weights="IMAGENET1K_V1")
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet, effnet_history = train_model(effnet, "casia_effnet")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 194MB/s]


Epoch 1: train_loss=0.5217 val_loss=0.4261 val_acc=0.7987 val_f1=0.6792 val_bal_acc=0.7572
Epoch 2: train_loss=0.3899 val_loss=0.3800 val_acc=0.8285 val_f1=0.7546 val_bal_acc=0.8160
Epoch 3: train_loss=0.3241 val_loss=0.3551 val_acc=0.8435 val_f1=0.7832 val_bal_acc=0.8409
Epoch 4: train_loss=0.2884 val_loss=0.3563 val_acc=0.8399 val_f1=0.7845 val_bal_acc=0.8445
Epoch 5: train_loss=0.2659 val_loss=0.3670 val_acc=0.8301 val_f1=0.7591 val_bal_acc=0.8201
Epoch 6: train_loss=0.2505 val_loss=0.3953 val_acc=0.8429 val_f1=0.7857 val_bal_acc=0.8442
Epoch 7: train_loss=0.2190 val_loss=0.3994 val_acc=0.8347 val_f1=0.7689 val_bal_acc=0.8288
Early stopping at epoch 7.


## Load existing checkpoints (skip retraining next time)
Loads the best saved checkpoint for each architecture without retraining. Run this instead of the two training cells above once checkpoints exist.

In [ ]:
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_resnet_best.pt", map_location=device))
resnet = resnet.to(device)
resnet.eval()

effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device)
effnet.eval()

print("Both checkpoints loaded successfully.")

Both checkpoints loaded successfully.


## Confusion matrix check
Confirms balanced recall across both classes (no majority-class collapse) for whichever model is currently loaded above.

In [13]:
model_to_check = resnet

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_to_check(images)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Authentic", "Tampered"]))

[[1109  174]
 [ 178  481]]
              precision    recall  f1-score   support

   Authentic       0.86      0.86      0.86      1283
    Tampered       0.73      0.73      0.73       659

    accuracy                           0.82      1942
   macro avg       0.80      0.80      0.80      1942
weighted avg       0.82      0.82      0.82      1942



In [15]:
model_to_check = effnet

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_to_check(images)
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=["Authentic", "Tampered"]))

[[1087  196]
 [ 125  534]]
              precision    recall  f1-score   support

   Authentic       0.90      0.85      0.87      1283
    Tampered       0.73      0.81      0.77       659

    accuracy                           0.83      1942
   macro avg       0.81      0.83      0.82      1942
weighted avg       0.84      0.83      0.84      1942



In [19]:
model_to_check = effnet


In [20]:
import numpy as np
print(f"Currently testing: {'ResNet18' if model_to_check is resnet else 'EfficientNet-B0' if model_to_check is effnet else 'UNKNOWN'}")

resnet_spliced_preds = []
with torch.no_grad():
    for i, fname in enumerate(spliced_files_all):
        img = Image.open(os.path.join(tp_dir, fname)).convert("RGB")
        input_tensor = val_transform(img).unsqueeze(0).to(device)
        pred = model_to_check(input_tensor).argmax(1).item()
        resnet_spliced_preds.append(pred)
        if (i + 1) % 200 == 0:
            print(f"Processed {i+1}/{len(spliced_files_all)}")

resnet_spliced_preds = np.array(resnet_spliced_preds)
print(f"Predicted Tampered on all spliced images: "
      f"{(resnet_spliced_preds==1).sum()}/{len(resnet_spliced_preds)} "
      f"({(resnet_spliced_preds==1).mean()*100:.1f}%)")

Currently testing: EfficientNet-B0
Processed 200/1828
Processed 400/1828
Processed 600/1828
Processed 800/1828
Processed 1000/1828
Processed 1200/1828
Processed 1400/1828
Processed 1600/1828
Processed 1800/1828
Predicted Tampered on all spliced images: 1522/1828 (83.3%)
